In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
from matplotlib.ticker import FuncFormatter, LogLocator,ScalarFormatter
from matplotlib.ticker import MultipleLocator
import matplotlib.lines as mlines
%run constants_functions.ipynb

bar = 1e5
sec_per_year = 60*60*24*365

plt.rcParams.update({
    # FIGURE & AXES SIZING
    "figure.figsize": (6, 4),  # Set figure size (width, height) in inches
    "figure.dpi": 300,  # High resolution for publication quality
    "axes.titlesize": 14,  # Title font size
    "axes.labelsize": 7,  # Axis label font size
    "axes.labelpad": 3,  # Padding for axis labels

    # TICKS & GRIDLINES
    "xtick.labelsize": 30,  # X-axis tick label font size
    "ytick.labelsize": 30,  # Y-axis tick label font size
    "xtick.major.size": 10,  # Major tick size
    "ytick.major.size": 10,
    "xtick.minor.size": 5,  # Minor tick size
    "ytick.minor.size": 5,
    "xtick.direction": "in",  # Ticks inside the plot
    "ytick.direction": "in",

    # GRID SETTINGS
    "axes.grid": True,  # Enable grid by default
    "grid.color": "gray",  # Grid line color
    "grid.linestyle": "--",  # Dashed grid lines for major ticks
    "grid.linewidth": 0.6,  # Thin grid lines
    "grid.alpha": 0.5,  # Slight transparency for better readability

    # MINOR GRID SETTINGS
    "grid.linestyle": ":",  # Dotted style for minor grid lines
    "grid.linewidth": 0.4,  # Thinner minor grid lines

    # LINES & MARKERS
    "lines.linewidth": 2,  # Line thickness
    "lines.markersize": 6,  # Marker size

    # FONT SETTINGS
    "font.family": "sans-serif",  # Use serif font (e.g., Times New Roman)
    "font.size": 12,  # General font size

    # LEGEND SETTINGS
    "legend.frameon": False,  # No legend border
    "legend.fontsize": 10,
    "legend.loc": "best",

    # SAVEFIG SETTINGS
    "savefig.dpi": 300,  # High resolution for saving figures
    "savefig.transparent": True,  # Transparent background for easy overlay in publications
    "savefig.bbox": "tight",  # Prevents cutting off labels when saving

    # AXES SPINES
    "axes.spines.top": True,  # Show top spine
    "axes.spines.right": True,  # Show right spine
    "axes.spines.left": True,
    "axes.spines.bottom": True,
})

In [ ]:
def solar_evolution(time):
    L_sun = 3.828e26 # W
    to = 4.57e9
    t_years = time*1
    Lum = L_sun*(1+(2/5)*(1-t_years/to))**(-1)
    S = Lum/(4*np.pi*a**2)
    return(S)

In [ ]:
## load all solutions
all_solutions = pd.read_pickle('../baseline/all_solutions.pkl')

In [ ]:
## sort solutions 

TL_all = all_solutions.loc[all_solutions['final state']=='TL']
TL_everhab = TL_all.loc[TL_all['ever hab?'] == 1]
TL_neverhab = TL_all.loc[TL_all['ever hab?'] == 0]

not_eq_all = all_solutions.loc[all_solutions['final state']=='not eq']
not_eq_neverhab = not_eq_all.loc[not_eq_all['ever hab?'] == 0]
not_eq_everhab = not_eq_all.loc[not_eq_all['ever hab?'] == 1]

Venus_all = all_solutions.loc[all_solutions['final state']=='Venus']
Venus_everhab = Venus_all.loc[Venus_all['ever hab?']==1]
Venus_neverhab = Venus_all.loc[Venus_all['ever hab?']==0]

altVenus_all = all_solutions.loc[all_solutions['final state']=='Prograde Venus']
altVenus_everhab = altVenus_all.loc[altVenus_all['ever hab?']==1]
altVenus_neverhab = altVenus_all.loc[altVenus_all['ever hab?']==0]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker

fig, ax = plt.subplots(2,1,figsize=(10, 9),tight_layout=True)



def rate_to_period(Omega, _):
    if Omega == 0:
        return("∞")
    period= 2*np.pi/Omega/(60*60*24)
    return(f"{period:.1f}")

period_ticks = np.array([-1,-10,-100,-400,np.inf,400,100,10,1])
period_ticks_top = np.array([400,100,10,1])
period_ticks_bottom = np.array([1,10,100,400])

omega_ticks = 2*np.pi/(period_ticks*60*60*24)
omega_ticks_top = 2*np.pi/(period_ticks_top*60*60*24)
omega_ticks_bottom = 2*np.pi/(period_ticks_bottom*60*60*24)
Venuscolor = 'darkorange'
altVenuscolor = 'skyblue'
TLcolor = 'silver'
noteqcolor = '#1b9e77'

varx ='$t_s$'
vary ='$T_o$'
## tidally locked
n=1
size=[30, 30, 30,50]
everhab = [TL_everhab, not_eq_everhab, altVenus_everhab, Venus_everhab]
neverhab = [TL_neverhab, not_eq_neverhab, altVenus_neverhab, Venus_neverhab]
prograde_zorder = [0, 1, 2, 3]
retrograde_zorder = [0,2,3,2]
everhab_alpha = 1
neverhab_alpha = 1
lw=0.5
div=2
everhab_line = 'k'
neverhab_line='r'
n_orb = 2*np.pi/(225*60*60*24)
colors = [TLcolor, noteqcolor, altVenuscolor, Venuscolor]

# Central plots--scatters
ax_center_top = ax[0]

ax_center_bottom = ax[1]



for i in range(len(everhab)):
    prograde_everhab = everhab[i].loc[everhab[i]['$\\epsilon_o$'] < 90]
    retrograde_everhab = everhab[i].loc[everhab[i]['$\\epsilon_o$'] >= 90]
    prograde_neverhab = neverhab[i].loc[neverhab[i]['$\\epsilon_o$'] < 90]
    retrograde_neverhab = neverhab[i].loc[neverhab[i]['$\\epsilon_o$'] >= 90]

    ax_center_top.scatter(prograde_everhab[varx][::n],prograde_everhab[vary][::n], facecolor = colors[i],edgecolors=everhab_line,lw=lw,alpha=everhab_alpha,s=size[i], zorder=prograde_zorder[i])
    ax_center_top.scatter(prograde_neverhab[varx][::n], prograde_neverhab[vary][::n], facecolor = colors[i],edgecolors=neverhab_line,lw=lw,alpha=neverhab_alpha,s=size[i], zorder=prograde_zorder[i])
    ax_center_bottom.scatter(retrograde_everhab[varx][::n], retrograde_everhab[vary][::n], facecolor = colors[i],edgecolors=everhab_line,lw=lw,alpha=everhab_alpha,s=size[i], zorder=retrograde_zorder[i])
    ax_center_bottom.scatter(retrograde_neverhab[varx][::n], retrograde_neverhab[vary][::n], facecolor = colors[i],edgecolors=neverhab_line,lw=lw,alpha=neverhab_alpha,s=size[i], zorder=retrograde_zorder[i])
    ax_center_top.set_ylim(400,0)
    ax_center_bottom.set_ylim(-0,-400)
    ax_center_top.get_xaxis().set_visible(False)


    #ax_center_bottom.set_yticks([-1,-50,-100,-150,-200,-250,-300,-350,-400])

    ax_center_top.set_xscale('log')
    ax_center_top.set_xticks([])
    ax_center_bottom.set_xscale('log')
    ax_center_bottom.set_xlim(9e5,4e9)
    ax_center_top.set_xlim(9e5,4e9)
    #plt.ylabel(labely)
    ax_center_bottom.set_xlabel('Duration of steam atmosphere state (yr)', fontsize=15)
    ax_center_top.grid(False)
    ax_center_bottom.grid(False)

    fig.text(-0.02, 0.5, 'Initial rotation period (days)', rotation = -270,va="center", fontsize=15)
    
    ax_center_top.tick_params(axis='both', which='major', labelsize=12)
    ax_center_bottom.tick_params(axis='both', which='major', labelsize=12)
    
    handles = [
        mlines.Line2D([], [], color=TLcolor, marker='o', linestyle='None', markersize=10, label='Tidally Locked'),
        mlines.Line2D([], [], color=noteqcolor, marker='o', linestyle='None', markersize=10, label='Not Equilibrated'),
        mlines.Line2D([], [], color=altVenuscolor, marker='o', linestyle='None', markersize=10, label='Prograde Venus'),
        mlines.Line2D([], [], color=Venuscolor, marker='o', linestyle='None', markersize=10, label='Modern Venus')
    ]

    # Add legend to top central plot (you could also add it to fig or bottom axis if you prefer)
    ax_center_top.legend(handles=handles, loc=(0,1.02), frameon=True, ncol=4, fontsize=12.6)


        
    ax_center_top.set_ylim(400,0.9)
    ax_center_top.set_yscale('log')#, linthresh=0.1e-10)
    ax_center_top.set_yticks(period_ticks_top)


    ax_center_bottom.set_ylim(0.9,400)
    ax_center_bottom.set_yscale('log')#, linthresh=0.1e-10)
    ax_center_bottom.set_yticks(period_ticks_bottom)
   
    ax_center_top.yaxis.set_major_formatter(ScalarFormatter())
    ax_center_top.yaxis.get_major_formatter().set_scientific(False)
    ax_center_top.set_yticks(period_ticks_top)
    ax_center_bottom.yaxis.set_major_formatter(ScalarFormatter())
    ax_center_bottom.yaxis.get_major_formatter().set_scientific(False)
    ax_center_bottom.set_yticks(period_ticks_bottom)
    
    #ax_center_bottom.scatter(sample_ts, sample_To, marker='*', color=Venuscolor, edgecolor='k', lw=1.5, s=300, zorder=10)
    
    ax_center_top.annotate('Initially prograde', xy = (8e8,300), bbox=dict(boxstyle="round", facecolor="w", alpha=1),fontsize=13)
    ax_center_bottom.annotate('Initially retrograde', xy = (7e8,250), bbox=dict(boxstyle="round", facecolor="w", alpha=1),fontsize=13)
#plt.savefig('Figure_2.png', bbox_inches='tight', dpi=150)
plt.show()
